In [0]:
from pyspark.sql import functions as F

df_judicial = spark.read.format("delta").load(
    "/Volumes/workspace/default/tfm_visnu_raw/clean/dataset_judicial_provincia_delta"
)

df_ine = spark.read.format("delta").load(
    "/Volumes/workspace/default/tfm_visnu_raw/clean/dataset_ine_base_delta"
)

display(df_judicial)
display(df_ine)

anio,provincia,denuncias,victimas,ordenes_proteccion,quebrantamientos
2014,null,42883.0,42883.0,11197.0,1170.0
2014,A CORUNA,37.0,37.0,15.0,3.0
2014,A CORUÑA,1478.0,1478.0,292.0,46.0
2014,ALAVA,43.0,43.0,11.0,0.0
2014,ALBACETE,774.0,774.0,319.0,29.0
2014,ALICANTE,7749.0,7749.0,2411.0,122.0
2014,ALMERIA,1672.0,1672.0,343.0,185.0
2014,ASTURIAS,1711.0,1711.0,472.0,45.0
2014,AVILA,266.0,266.0,117.0,0.0
2014,BADAJOZ,1417.0,1417.0,509.0,17.0


anio,provincia,poblacion_hombres,poblacion_mujeres,poblacion_total,renta_media_hogar,renta_media_persona,tasa_actividad_total,tasa_actividad_hombres,tasa_actividad_mujeres,tasa_empleo_total,tasa_empleo_hombres,tasa_empleo_mujeres,tasa_paro_total,tasa_paro_hombres,tasa_paro_mujeres
2021,ALBACETE,193487.0,193239.0,386726.0,29451.0,11652.0,57.052499999999995,63.6175,50.55499999999999,48.165,56.010000000000005,40.4075,15.5275,11.9675,19.91
2021,ALICANTE,935234.0,951802.0,1887036.0,27380.0,10770.0,56.585,62.307500000000005,51.015,46.00000000000001,52.14,40.0275,18.705,16.325,21.5225
2021,ALMERIA,372823.0,357607.0,730430.0,27184.0,10103.0,61.765,69.485,53.765,49.89,57.025,42.4875,19.255000000000003,17.97,20.9875
2021,ALAVA,164299.0,169186.0,333485.0,36546.0,15539.0,57.25749999999999,61.7725,52.942499999999995,50.755,54.29,47.37499999999999,11.365,12.129999999999999,10.522499999999999
2021,ASTURIAS,483186.0,528931.0,1012117.0,31623.0,14057.0,50.412499999999994,55.3125,46.027499999999996,44.19,49.055,39.8375,12.335,11.317499999999999,13.4275
2021,AVILA,79884.0,79014.0,158898.0,27023.0,12123.0,52.325,57.545,47.06,44.2,51.68749999999999,36.655,15.49,10.1675,22.03
2021,BADAJOZ,331856.0,339236.0,671092.0,26050.0,10549.0,55.74499999999999,62.4475,49.287499999999994,44.32,52.62499999999999,36.3175,20.5275,15.7625,26.332499999999996
2021,ILLES BALEARS,590990.0,592425.0,1183415.0,36905.0,13468.0,63.215,67.5775,58.92,54.16,58.66,49.73,14.435,13.307500000000001,15.7
2021,BARCELONA,2785890.0,2916372.0,5702262.0,39511.0,15297.0,61.370000000000005,65.91,57.1325,54.4225,59.2025,49.957499999999996,11.327499999999999,10.1775,12.5625
2021,BIZKAIA,557190.0,596445.0,1153635.0,38390.0,16192.0,55.4,59.9825,51.182500000000005,49.2825,52.9675,45.8925,11.065000000000001,11.715,10.3575


In [0]:
print("Años judicial:")
display(df_judicial.select("anio").distinct().orderBy("anio"))

print("Años INE:")
display(df_ine.select("anio").distinct().orderBy("anio"))

Años judicial:


anio
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023


Años INE:


anio
2021
2022
2023


In [0]:
df_judicial = df_judicial.filter(F.col("anio").between(2021, 2024))

display(df_judicial.select("anio").distinct().orderBy("anio"))

anio
2021
2022
2023
2024


In [0]:
from pyspark.sql import functions as F

def normalizar_provincia(df, col_name="provincia"):
    return (
        df
        .filter(F.col(col_name).isNotNull())
        .withColumn(col_name, F.upper(F.trim(F.col(col_name))))
        .withColumn(col_name, F.regexp_replace(col_name, "Á", "A"))
        .withColumn(col_name, F.regexp_replace(col_name, "É", "E"))
        .withColumn(col_name, F.regexp_replace(col_name, "Í", "I"))
        .withColumn(col_name, F.regexp_replace(col_name, "Ó", "O"))
        .withColumn(col_name, F.regexp_replace(col_name, "Ú", "U"))
        .withColumn(col_name, F.regexp_replace(col_name, "Ü", "U"))
        .withColumn(col_name, F.regexp_replace(col_name, "Ñ", "N"))
        .withColumn(col_name, F.regexp_replace(col_name, r"\s+", " "))
    )

df_judicial_norm = normalizar_provincia(df_judicial, "provincia")
df_ine_norm = normalizar_provincia(df_ine, "provincia")

In [0]:
print("Provincias judicial que no están en INE")
display(
    df_judicial_norm
    .select("provincia")
    .distinct()
    .join(
        df_ine_norm.select("provincia").distinct(),
        on="provincia",
        how="left_anti"
    )
    .orderBy("provincia")
)

print("Provincias INE que no están en judicial")
display(
    df_ine_norm
    .select("provincia")
    .distinct()
    .join(
        df_judicial_norm.select("provincia").distinct(),
        on="provincia",
        how="left_anti"
    )
    .orderBy("provincia")
)

Provincias judicial que no están en INE


provincia


Provincias INE que no están en judicial


provincia


In [0]:
display(
    df_ine_norm.selectExpr(
        "count(*) as filas",
        "count(distinct provincia) as provincias",
        "count(distinct anio) as anios",
        "sum(case when renta_media_persona is null then 1 else 0 end) as null_renta",
        "sum(case when poblacion_total is null then 1 else 0 end) as null_poblacion",
        "sum(case when tasa_paro_mujeres is null then 1 else 0 end) as null_paro"
    )
)

filas,provincias,anios,null_renta,null_poblacion,null_paro
156,52,3,0,0,0


In [0]:
## merge final de datasets
df_analitico = (
    df_judicial_norm
    .join(df_ine_norm, on=["anio", "provincia"], how="inner")
    .orderBy("anio", "provincia")
)

display(df_analitico)

anio,provincia,denuncias,victimas,ordenes_proteccion,quebrantamientos,poblacion_hombres,poblacion_mujeres,poblacion_total,renta_media_hogar,renta_media_persona,tasa_actividad_total,tasa_actividad_hombres,tasa_actividad_mujeres,tasa_empleo_total,tasa_empleo_hombres,tasa_empleo_mujeres,tasa_paro_total,tasa_paro_hombres,tasa_paro_mujeres
2021,A CORUNA,54.0,54.0,24.0,11.0,537361.0,583140.0,1120501.0,32643.0,13319.0,52.70249999999999,57.724999999999994,48.167500000000004,47.122499999999995,51.915,42.795,10.6,10.094999999999999,11.145
2021,A CORUNA,1943.0,1916.0,700.0,241.0,537361.0,583140.0,1120501.0,32643.0,13319.0,52.70249999999999,57.724999999999994,48.167500000000004,47.122499999999995,51.915,42.795,10.6,10.094999999999999,11.145
2021,ALAVA,53.0,53.0,5.0,1.0,164299.0,169186.0,333485.0,36546.0,15539.0,57.25749999999999,61.7725,52.942499999999995,50.755,54.29,47.37499999999999,11.365,12.129999999999999,10.522499999999999
2021,ALBACETE,969.0,956.0,336.0,108.0,193487.0,193239.0,386726.0,29451.0,11652.0,57.052499999999995,63.6175,50.55499999999999,48.165,56.010000000000005,40.4075,15.5275,11.9675,19.91
2021,ALICANTE,10659.0,10627.0,2984.0,1433.0,935234.0,951802.0,1887036.0,27380.0,10770.0,56.585,62.307500000000005,51.015,46.00000000000001,52.14,40.0275,18.705,16.325,21.5225
2021,ALMERIA,2433.0,2218.0,580.0,348.0,372823.0,357607.0,730430.0,27184.0,10103.0,61.765,69.485,53.765,49.89,57.025,42.4875,19.255000000000003,17.97,20.9875
2021,ASTURIAS,1578.0,1571.0,421.0,246.0,483186.0,528931.0,1012117.0,31623.0,14057.0,50.412499999999994,55.3125,46.027499999999996,44.19,49.055,39.8375,12.335,11.317499999999999,13.4275
2021,AVILA,374.0,374.0,141.0,42.0,79884.0,79014.0,158898.0,27023.0,12123.0,52.325,57.545,47.06,44.2,51.68749999999999,36.655,15.49,10.1675,22.03
2021,BADAJOZ,1970.0,1970.0,650.0,130.0,331856.0,339236.0,671092.0,26050.0,10549.0,55.74499999999999,62.4475,49.287499999999994,44.32,52.62499999999999,36.3175,20.5275,15.7625,26.332499999999996
2021,BARCELONA,11536.0,11262.0,2780.0,1335.0,2785890.0,2916372.0,5702262.0,39511.0,15297.0,61.370000000000005,65.91,57.1325,54.4225,59.2025,49.957499999999996,11.327499999999999,10.1775,12.5625


In [0]:
display(
    df_analitico.selectExpr(
        "count(*) as num_filas",
        "count(distinct provincia) as num_provincias",
        "min(anio) as anio_min",
        "max(anio) as anio_max",
        "sum(case when renta_media_persona is null then 1 else 0 end) as null_renta_persona",
        "sum(case when tasa_paro_mujeres is null then 1 else 0 end) as null_tasa_paro_mujeres",
        "sum(case when poblacion_total is null then 1 else 0 end) as null_poblacion_total"
    )
)

num_filas,num_provincias,anio_min,anio_max,null_renta_persona,null_tasa_paro_mujeres,null_poblacion_total
159,52,2021,2023,0,0,0


In [0]:
display(
    df_analitico.selectExpr(
        "count(*) as num_filas",
        "count(distinct provincia) as num_provincias",
        "min(anio) as anio_min",
        "max(anio) as anio_max"
    )
)

num_filas,num_provincias,anio_min,anio_max
159,52,2021,2023


In [0]:
df_analitico = (
    df_analitico
    .withColumn("tasa_denuncias_100k", (F.col("denuncias") / F.col("poblacion_total")) * 100000)
    .withColumn("tasa_victimas_100k", (F.col("victimas") / F.col("poblacion_total")) * 100000)
    .withColumn("tasa_ordenes_100k", (F.col("ordenes_proteccion") / F.col("poblacion_total")) * 100000)
    .withColumn("tasa_quebrantamientos_100k", (F.col("quebrantamientos") / F.col("poblacion_total")) * 100000)
    .withColumn(
        "ratio_ordenes_denuncias",
        F.when(F.col("denuncias") > 0, F.col("ordenes_proteccion") / F.col("denuncias"))
    )
    .withColumn(
        "ratio_quebrantamientos_denuncias",
        F.when(F.col("denuncias") > 0, F.col("quebrantamientos") / F.col("denuncias"))
    )
    .withColumn(
        "porcentaje_mujeres",
        F.when(F.col("poblacion_total") > 0, F.col("poblacion_mujeres") / F.col("poblacion_total"))
    )
)

In [0]:
#validamos nulos

display(
    df_analitico.selectExpr(
        "sum(case when tasa_denuncias_100k is null then 1 else 0 end) as null_tasa_denuncias",
        "sum(case when renta_media_persona is null then 1 else 0 end) as null_renta_persona",
        "sum(case when tasa_paro_mujeres is null then 1 else 0 end) as null_tasa_paro_mujeres",
        "sum(case when poblacion_total is null then 1 else 0 end) as null_poblacion_total"
    )
)

null_tasa_denuncias,null_renta_persona,null_tasa_paro_mujeres,null_poblacion_total
0,0,0,0


In [0]:
display(df_analitico)

anio,provincia,denuncias,victimas,ordenes_proteccion,quebrantamientos,poblacion_hombres,poblacion_mujeres,poblacion_total,renta_media_hogar,renta_media_persona,tasa_actividad_total,tasa_actividad_hombres,tasa_actividad_mujeres,tasa_empleo_total,tasa_empleo_hombres,tasa_empleo_mujeres,tasa_paro_total,tasa_paro_hombres,tasa_paro_mujeres,tasa_denuncias_100k,tasa_victimas_100k,tasa_ordenes_100k,tasa_quebrantamientos_100k,ratio_ordenes_denuncias,ratio_quebrantamientos_denuncias,porcentaje_mujeres
2021,A CORUNA,54.0,54.0,24.0,11.0,537361.0,583140.0,1120501.0,32643.0,13319.0,52.70249999999999,57.724999999999994,48.167500000000004,47.122499999999995,51.915,42.795,10.6,10.094999999999999,11.145,4.819272807431675,4.819272807431675,2.1418990255251895,0.9817037200323784,0.4444444444444444,0.2037037037037037,0.5204279157269829
2021,A CORUNA,1943.0,1916.0,700.0,241.0,537361.0,583140.0,1120501.0,32643.0,13319.0,52.70249999999999,57.724999999999994,48.167500000000004,47.122499999999995,51.915,42.795,10.6,10.094999999999999,11.145,173.4045752748101,170.99493887109426,62.472054911151346,21.50823604798211,0.3602676273803397,0.1240349974266598,0.5204279157269829
2021,ALAVA,53.0,53.0,5.0,1.0,164299.0,169186.0,333485.0,36546.0,15539.0,57.25749999999999,61.7725,52.942499999999995,50.755,54.29,47.37499999999999,11.365,12.129999999999999,10.522499999999999,15.89276879020046,15.89276879020046,1.4993178103962697,0.29986356207925396,0.09433962264150944,0.018867924528301886,0.5073271661394065
2021,ALBACETE,969.0,956.0,336.0,108.0,193487.0,193239.0,386726.0,29451.0,11652.0,57.052499999999995,63.6175,50.55499999999999,48.165,56.010000000000005,40.4075,15.5275,11.9675,19.91,250.56499950869608,247.20344636771256,86.88321964388223,27.92674917124786,0.34674922600619196,0.11145510835913312,0.49967935954655235
2021,ALICANTE,10659.0,10627.0,2984.0,1433.0,935234.0,951802.0,1887036.0,27380.0,10770.0,56.585,62.307500000000005,51.015,46.00000000000001,52.14,40.0275,18.705,16.325,21.5225,564.854088634239,563.1583075256647,158.1315883745726,75.93919776835206,0.27995121493573505,0.13444037902242237,0.5043899533448223
2021,ALMERIA,2433.0,2218.0,580.0,348.0,372823.0,357607.0,730430.0,27184.0,10103.0,61.765,69.485,53.765,49.89,57.025,42.4875,19.255000000000003,17.97,20.9875,333.09146667031746,303.656750133483,79.40528182029763,47.64316909217858,0.23838882038635428,0.14303329223181258,0.4895842175157099
2021,ASTURIAS,1578.0,1571.0,421.0,246.0,483186.0,528931.0,1012117.0,31623.0,14057.0,50.412499999999994,55.3125,46.027499999999996,44.19,49.055,39.8375,12.335,11.317499999999999,13.4275,155.91082849117245,155.21920884640807,41.5959814922583,24.305490373148558,0.26679340937896073,0.155893536121673,0.5225986718926764
2021,AVILA,374.0,374.0,141.0,42.0,79884.0,79014.0,158898.0,27023.0,12123.0,52.325,57.545,47.06,44.2,51.68749999999999,36.655,15.49,10.1675,22.03,235.37111857921434,235.37111857921434,88.73617037344711,26.432050749537442,0.3770053475935829,0.11229946524064172,0.4972623947437979
2021,BADAJOZ,1970.0,1970.0,650.0,130.0,331856.0,339236.0,671092.0,26050.0,10549.0,55.74499999999999,62.4475,49.287499999999994,44.32,52.62499999999999,36.3175,20.5275,15.7625,26.332499999999996,293.55140576850863,293.55140576850863,96.85706281702062,19.37141256340412,0.3299492385786802,0.06598984771573604,0.5054985009506894
2021,BARCELONA,11536.0,11262.0,2780.0,1335.0,2785890.0,2916372.0,5702262.0,39511.0,15297.0,61.370000000000005,65.91,57.1325,54.4225,59.2025,49.957499999999996,11.327499999999999,10.1775,12.5625,202.30568149972763,197.50057082610377,48.75258274698707,23.411761858715014,0.24098474341192788,0.1157246879334258,0.5114412491043028


In [0]:
df_analitico.write.mode("overwrite").format("delta").save(
    "/Volumes/workspace/default/tfm_visnu_raw/preprocess/dataset_analitico_final_delta"
)

In [0]:
display(
    df_analitico.selectExpr(
        "count(*) as num_filas",
        "count(distinct provincia) as num_provincias",
        "min(anio) as anio_min",
        "max(anio) as anio_max"
    )
)

num_filas,num_provincias,anio_min,anio_max
159,52,2021,2023


In [0]:
display(
    df_analitico
    .groupBy("provincia", "anio")
    .count()
    .filter(F.col("count") > 1)
)

provincia,anio,count
A CORUNA,2021,2
A CORUNA,2022,2
A CORUNA,2023,2


In [0]:
df_analitico = (
    df_analitico
    .groupBy("provincia", "anio")
    .agg(
        F.first("denuncias").alias("denuncias"),
        F.first("victimas").alias("victimas"),
        F.first("ordenes_proteccion").alias("ordenes_proteccion"),
        F.first("quebrantamientos").alias("quebrantamientos"),
        F.first("poblacion_total").alias("poblacion_total"),
        F.first("poblacion_mujeres").alias("poblacion_mujeres"),
        F.first("renta_media_hogar").alias("renta_media_hogar"),
        F.first("renta_media_persona").alias("renta_media_persona"),
        F.first("tasa_actividad_total").alias("tasa_actividad_total"),
        F.first("tasa_actividad_hombres").alias("tasa_actividad_hombres"),
        F.first("tasa_actividad_mujeres").alias("tasa_actividad_mujeres"),
        F.first("tasa_empleo_total").alias("tasa_empleo_total"),
        F.first("tasa_empleo_hombres").alias("tasa_empleo_hombres"),
        F.first("tasa_empleo_mujeres").alias("tasa_empleo_mujeres"),
        F.first("tasa_paro_total").alias("tasa_paro_total"),
        F.first("tasa_paro_hombres").alias("tasa_paro_hombres"),
        F.first("tasa_paro_mujeres").alias("tasa_paro_mujeres")
    )
)

In [0]:
display(
    df_analitico.selectExpr(
        "count(*) as num_filas",
        "count(distinct provincia) as num_provincias",
        "min(anio) as anio_min",
        "max(anio) as anio_max",
        "sum(case when renta_media_persona is null then 1 else 0 end) as null_renta_persona",
        "sum(case when tasa_paro_mujeres is null then 1 else 0 end) as null_tasa_paro_mujeres",
        "sum(case when poblacion_total is null then 1 else 0 end) as null_poblacion_total"
    )
)

num_filas,num_provincias,anio_min,anio_max,null_renta_persona,null_tasa_paro_mujeres,null_poblacion_total
156,52,2021,2023,0,0,0


In [0]:
df_analitico.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico/_committed_184625871374323417,_committed_184625871374323417,211,1777329308000
dbfs:/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico/_committed_8877462158814190759,_committed_8877462158814190759,200,1777332377000
dbfs:/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico/_started_8877462158814190759,_started_8877462158814190759,0,1777332376000
dbfs:/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico/part-00000-tid-8877462158814190759-515138d0-408b-48f3-8f44-4766c7c2c03b-578-1-c000.csv,part-00000-tid-8877462158814190759-515138d0-408b-48f3-8f44-4766c7c2c03b-578-1-c000.csv,26207,1777332377000


In [0]:
df_analitico.write \
    .mode("overwrite") \
    .saveAsTable("default.df_analitico")

In [0]:
df_analitico.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico_final")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico_final"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico_final/_committed_2952115847571671587,_committed_2952115847571671587,201,1777332384000
dbfs:/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico_final/_committed_8925415772557031463,_committed_8925415772557031463,212,1777329319000
dbfs:/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico_final/_started_2952115847571671587,_started_2952115847571671587,0,1777332383000
dbfs:/Volumes/workspace/default/tfm_visnu_raw/exports/df_analitico_final/part-00000-tid-2952115847571671587-f02dc33a-9991-41e6-839d-bf0b1607666a-590-1-c000.csv,part-00000-tid-2952115847571671587-f02dc33a-9991-41e6-839d-bf0b1607666a-590-1-c000.csv,26207,1777332383000
